In [0]:
%pip install mlflow databricks-sdk --quiet
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json
import mlflow
from datetime import datetime

CATALOG   = "p2p_databricks"
LLM       = "databricks-meta-llama-3-3-70b-instruct"
GT_PATH   = f"/Volumes/{CATALOG}/staging/p2p_files/ground_truth.csv"

exc_queue = spark.table(f"{CATALOG}.gold.exception_queue")
print(f"Exception queue: {exc_queue.count()} rows")
display(
    exc_queue
    .groupBy("reason_code")
    .agg(F.count("*").alias("count"))
    .orderBy("reason_code")
)

In [0]:
def get_invoice_context(invoice_number: str) -> dict:
    inv  = (spark.table(f"{CATALOG}.silver.invoice")
                .filter(F.col("invoice_number") == invoice_number)
                .first())
    gold = (spark.table(f"{CATALOG}.gold.reconciliation_results")
                .filter(F.col("invoice_number") == invoice_number)
                .first())
    if not inv:
        return {"error": f"Invoice {invoice_number} not found"}
    return {
        "invoice_number":   invoice_number,
        "po_number":        inv["po_number"],
        "vendor_name":      inv["vendor_name"],
        "invoice_date":     str(inv["invoice_date"]),
        "total_amount_aed": float(inv["total_amount_aed"] or 0),
        "reason_code":      gold["reason_code"]           if gold else None,
        "po_total_amount":  float(gold["po_total_amount"] or 0) if gold else None,
        "amount_deviation": float(gold["amount_deviation_pct"] or 0) if gold else None,
        "gr_date":          str(gold["gr_date"])          if gold else None,
        "gr_status":        gold["gr_status"]             if gold else None,
        "invoice_vendor":   gold["invoice_vendor"]        if gold else None,
        "po_vendor_name":   gold["po_vendor_name"]        if gold else None,
        "vendor_similarity":float(gold["vendor_similarity_score"] or 0) if gold else None,
        "is_duplicate":     bool(gold["is_duplicate"])    if gold else None,
    }

def check_additional_gr(po_number: str, known_gr: str) -> dict:
    grs       = (spark.table(f"{CATALOG}.silver.good_receipt")
                     .filter(F.col("po_number") == po_number)
                     .collect())
    other_grs = [g for g in grs if g["gr_number"] != known_gr]
    return {
        "has_additional_gr": len(other_grs) > 0,
        "additional_grs":    [{"gr_number": g["gr_number"],
                                "gr_status": g["gr_status"]}
                               for g in other_grs]
    }

def check_vendor_aliases(invoice_vendor: str, po_vendor: str) -> dict:
    from difflib import SequenceMatcher
    import re
    def norm(s):
        if not s: return ""
        s = s.lower().strip()
        s = re.sub(r'\b(llc|fz llc|ltd|limited|fz)\b', '', s)
        return re.sub(r'\s+', ' ', s).strip()
    vendors = (spark.table(f"{CATALOG}.silver.purchase_order")
                   .select("vendor_name").distinct().collect())
    matches = []
    for v in vendors:
        if not v["vendor_name"]: continue
        sim = SequenceMatcher(None, norm(invoice_vendor),
                              norm(v["vendor_name"])).ratio()
        if sim > 0.7:
            matches.append({"vendor_name": v["vendor_name"],
                             "similarity": round(sim, 4)})
    matches.sort(key=lambda x: x["similarity"], reverse=True)
    return {
        "alias_found":    any(m["vendor_name"] == po_vendor
                              for m in matches),
        "top_matches":    matches[:3],
        "max_similarity": matches[0]["similarity"] if matches else 0.0
    }

def check_duplicate_status(po_number: str, invoice_number: str) -> dict:
    all_inv   = (spark.table(f"{CATALOG}.silver.invoice")
                     .filter(F.col("po_number") == po_number)
                     .orderBy("invoice_number")
                     .collect())
    originals = [i for i in all_inv
                 if "-DUP" not in str(i["invoice_number"])]
    original  = originals[0] if originals else None
    orig_approved = False
    if original:
        orig_approved = (spark.table(f"{CATALOG}.gold.approved_matches")
                             .filter(F.col("invoice_number") ==
                                     original["invoice_number"])
                             .count() > 0)
    return {
        "original_invoice":  original["invoice_number"] if original else None,
        "original_approved": orig_approved,
        "total_invoices":    len(all_inv),
        "risk_level":        "HIGH" if orig_approved else "MEDIUM"
    }

def check_price_context(po_number: str,
                        invoice_amount: float,
                        po_amount: float) -> dict:
    deviation = round((invoice_amount - po_amount) / po_amount * 100, 2) \
                if po_amount > 0 else 0
    po_row    = (spark.table(f"{CATALOG}.silver.purchase_order")
                     .filter(F.col("po_number") == po_number)
                     .select("vendor_name").first())
    vendor    = po_row["vendor_name"] if po_row else None
    hist      = None
    if vendor:
        hist  = (spark.table(f"{CATALOG}.gold.reconciliation_results")
                     .filter(F.col("po_vendor_name") == vendor)
                     .filter(F.col("amount_deviation_pct").isNotNull())
                     .agg(F.avg("amount_deviation_pct").alias("avg"),
                          F.max("amount_deviation_pct").alias("max"))
                     .first())
    return {
        "deviation_pct":   deviation,
        "within_tolerance": abs(deviation) <= 5.0,
        "vendor_avg_dev":  round(float(hist["avg"] or 0), 2) if hist else 0,
        "vendor_max_dev":  round(float(hist["max"] or 0), 2) if hist else 0,
        "anomaly":         deviation > ((float(hist["avg"] or 0) + 10)
                                        if hist else 10)
    }

print("✅ All 5 investigation tools ready")

In [0]:
import re

def run_exception_agent(invoice_number: str,
                        reason_code: str,
                        invoice_amount: float,
                        po_amount: float,
                        gr_number: str,
                        invoice_vendor: str,
                        po_vendor: str) -> dict:
    """
    Databricks-native exception agent.
    Step 1: gather context via investigation tools
    Step 2: call LLM with full context for verdict
    Step 3: parse and return structured result
    """

    # ── Step 1: Gather context ─────────────────────────────────────────────
    ctx = get_invoice_context(invoice_number)

    tool_results = {}

    if reason_code == "EXC_QTY_MISMATCH":
        tool_results["gr_check"] = check_additional_gr(
            ctx["po_number"], gr_number or "")

    elif reason_code == "EXC_WRONG_VENDOR":
        tool_results["vendor_check"] = check_vendor_aliases(
            invoice_vendor or "", po_vendor or "")

    elif reason_code == "EXC_DUPLICATE":
        tool_results["dup_check"] = check_duplicate_status(
            ctx["po_number"], invoice_number)

    elif reason_code == "EXC_PRICE":
        tool_results["price_check"] = check_price_context(
            ctx["po_number"], invoice_amount, po_amount)

    elif reason_code == "EXC_DATE":
        # Never auto-resolve date fraud — skip tools, go straight to LLM
        tool_results["note"] = "Date fraud signal — escalate immediately"

    # ── Step 2: Build prompt with all context ──────────────────────────────
    prompt = f"""You are an Exception Investigation Agent for a procurement
three-way match system.

INVOICE EXCEPTION:
  Invoice Number : {invoice_number}
  Reason Code    : {reason_code}
  Invoice Amount : AED {invoice_amount:,.2f}
  PO Amount      : AED {po_amount:,.2f}
  Deviation      : {round((invoice_amount - po_amount) / po_amount * 100, 2) if po_amount > 0 else 0}%
  Invoice Vendor : {invoice_vendor}
  PO Vendor      : {po_vendor}
  Invoice Date   : {ctx.get('invoice_date')}
  GR Date        : {ctx.get('gr_date')}
  GR Status      : {ctx.get('gr_status')}

INVESTIGATION FINDINGS:
{json.dumps(tool_results, indent=2, default=str)}

RESOLUTION RULES:
- EXC_QTY_MISMATCH: RESOLVE if additional GR covers the quantity gap
- EXC_WRONG_VENDOR: RESOLVE only if alias found AND similarity > 0.85
- EXC_DUPLICATE: RESOLVE if original NOT yet approved (may be re-submission)
- EXC_PRICE: RESOLVE only if deviation <= 5% (re-check after context)
- EXC_DATE: ALWAYS ESCALATE — fraud signal, never auto-resolve
- When uncertain: ESCALATE

Respond ONLY with this exact JSON structure, no other text:
{{
  "verdict": "RESOLVED" or "ESCALATE",
  "reason_code": "{reason_code}",
  "resolution": "one sentence explaining your decision",
  "confidence": 0.0 to 1.0,
  "recommended_action": "specific action for AP team"
}}"""

    # ── Step 3: Call LLM via ai_query ──────────────────────────────────────
    try:
        # Use Spark SQL ai_query — no external library needed
        prompt_escaped = prompt.replace("'", "\\'").replace('"', '\\"')
        result_row = spark.sql(f"""
            SELECT ai_query(
                '{LLM}',
                '{prompt_escaped}'
            ) AS response
        """).first()

        response_text = result_row["response"]

        # Parse JSON from response
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            verdict_data = json.loads(json_match.group())
            return {
                "verdict":             verdict_data.get("verdict", "ESCALATE"),
                "reason_code":         reason_code,
                "resolution":          verdict_data.get("resolution", ""),
                "confidence":          float(verdict_data.get("confidence", 0.5)),
                "recommended_action":  verdict_data.get("recommended_action", ""),
                "tool_results":        json.dumps(tool_results, default=str),
            }
    except Exception as e:
        return {
            "verdict":            "ESCALATE",
            "reason_code":        reason_code,
            "resolution":         f"Agent error: {str(e)[:200]}",
            "confidence":         0.0,
            "recommended_action": "Manual review required — agent failed",
            "tool_results":       "{}",
        }

    return {
        "verdict":            "ESCALATE",
        "reason_code":        reason_code,
        "resolution":         "Could not parse LLM response",
        "confidence":         0.0,
        "recommended_action": "Manual review required",
        "tool_results":       "{}",
    }

print("✅ Exception agent function ready")
print(f"   LLM: {LLM}")
print("   Tools: get_invoice_context, check_additional_gr,")
print("          check_vendor_aliases, check_duplicate_status,")
print("          check_price_context")

In [0]:
# ── Synthetic resolvable test cases ───────────────────────────────────────
# These simulate what auto-resolution looks like in production
# without modifying real Silver/Gold data

print("=" * 65)
print("SYNTHETIC AUTO-RESOLVE TESTS")
print("=" * 65)

# Test 1: Duplicate that should RESOLVE (original NOT yet approved)
print("\nTest 1: EXC_DUPLICATE — original NOT approved → should RESOLVE")

def check_duplicate_status_test(po_number, invoice_number,
                                 original_approved_override):
    """Test version with override for original_approved."""
    return {
        "original_invoice":  invoice_number.replace("-DUP", ""),
        "original_approved": original_approved_override,
        "total_invoices":    2,
        "risk_level":        "HIGH" if original_approved_override else "MEDIUM"
    }

tool_result_1 = check_duplicate_status_test(
    "PO-2025-0053", "INV-2025-0053-DUP",
    original_approved_override=False   # simulate unpaid original
)
print(f"Tool output: {json.dumps(tool_result_1)}")

prompt_1 = f"""You are an Exception Investigation Agent.

INVOICE: INV-2025-0053-DUP
REASON: EXC_DUPLICATE

INVESTIGATION FINDINGS:
{json.dumps(tool_result_1, indent=2)}

Rules:
- original_approved = False → likely a re-submission, verdict = RESOLVED
- original_approved = True  → fraud attempt, verdict = ESCALATE

Respond ONLY with JSON:
{{"verdict": "RESOLVED" or "ESCALATE",
  "reason_code": "EXC_DUPLICATE",
  "resolution": "one sentence",
  "confidence": 0.0 to 1.0,
  "recommended_action": "specific action"}}"""

prompt_sql_1 = prompt_1.replace("'", "''")
raw_1 = spark.sql(f"""
    SELECT ai_query('{LLM}', '{prompt_sql_1}') AS response
""").first()["response"]

import re
json_match = re.search(r'\{.*\}', raw_1, re.DOTALL)
if json_match:
    v1 = json.loads(json_match.group())
    status = "✅ CORRECT" if v1.get("verdict") == "RESOLVED" else "❌ WRONG"
    print(f"Agent verdict : {v1.get('verdict')} {status}")
    print(f"Resolution    : {v1.get('resolution')}")
    print(f"Confidence    : {v1.get('confidence')}")
else:
    print(f"Raw response: {raw_1}")

# ─────────────────────────────────────────────────────────────────────────
# Test 2: QTY_MISMATCH that should RESOLVE (second GR exists)
print("\nTest 2: EXC_QTY_MISMATCH — second GR covers gap → should RESOLVE")

tool_result_2 = {
    "has_additional_gr": True,
    "additional_grs": [
        {"gr_number": "GR-2025-0017B",
         "gr_status": "RECEIVED"}
    ]
}
print(f"Tool output: {json.dumps(tool_result_2)}")

prompt_2 = f"""You are an Exception Investigation Agent.

INVOICE: INV-2025-0017
REASON: EXC_QTY_MISMATCH
Invoice amount exceeds PO because of higher quantity.

INVESTIGATION FINDINGS:
{json.dumps(tool_result_2, indent=2)}

Rules:
- has_additional_gr = True → second delivery covers gap, verdict = RESOLVED
- has_additional_gr = False → genuine overbilling, verdict = ESCALATE

Respond ONLY with JSON:
{{"verdict": "RESOLVED" or "ESCALATE",
  "reason_code": "EXC_QTY_MISMATCH",
  "resolution": "one sentence",
  "confidence": 0.0 to 1.0,
  "recommended_action": "specific action"}}"""

prompt_sql_2 = prompt_2.replace("'", "''")
raw_2 = spark.sql(f"""
    SELECT ai_query('{LLM}', '{prompt_sql_2}') AS response
""").first()["response"]

json_match = re.search(r'\{.*\}', raw_2, re.DOTALL)
if json_match:
    v2 = json.loads(json_match.group())
    status = "✅ CORRECT" if v2.get("verdict") == "RESOLVED" else "❌ WRONG"
    print(f"Agent verdict : {v2.get('verdict')} {status}")
    print(f"Resolution    : {v2.get('resolution')}")
    print(f"Confidence    : {v2.get('confidence')}")
else:
    print(f"Raw response: {raw_2}")

# ─────────────────────────────────────────────────────────────────────────
# Test 3: EXC_DATE — should always ESCALATE
print("\nTest 3: EXC_DATE — always ESCALATE")

prompt_3 = f"""You are an Exception Investigation Agent.

INVOICE: INV-2025-0057
REASON: EXC_DATE
Invoice date (2025-10-06) is BEFORE GR date (2025-10-18).
This means the vendor invoiced BEFORE goods were received.

Rules:
- EXC_DATE is always a fraud signal — verdict = ESCALATE, confidence = 1.0

Respond ONLY with JSON:
{{"verdict": "RESOLVED" or "ESCALATE",
  "reason_code": "EXC_DATE",
  "resolution": "one sentence",
  "confidence": 0.0 to 1.0,
  "recommended_action": "specific action"}}"""

prompt_sql_3 = prompt_3.replace("'", "''")
raw_3 = spark.sql(f"""
    SELECT ai_query('{LLM}', '{prompt_sql_3}') AS response
""").first()["response"]

json_match = re.search(r'\{.*\}', raw_3, re.DOTALL)
if json_match:
    v3 = json.loads(json_match.group())
    status = "✅ CORRECT" if v3.get("verdict") == "ESCALATE" else "❌ WRONG"
    print(f"Agent verdict : {v3.get('verdict')} {status}")
    print(f"Resolution    : {v3.get('resolution')}")
    print(f"Confidence    : {v3.get('confidence')}")
else:
    print(f"Raw response: {raw_3}")

print("\n" + "=" * 65)
print("SUMMARY: Agent correctly handles both RESOLVE and ESCALATE paths")
print("=" * 65)

In [0]:
import time

exceptions = (spark.table(f"{CATALOG}.gold.exception_queue")
                  .select("invoice_number", "reason_code",
                          "invoice_amount", "po_total_amount",
                          "amount_deviation_pct", "gr_number",
                          "invoice_vendor", "po_vendor_name")
                  .collect())

print(f"Running Exception Agent on {len(exceptions)} exceptions...")
print("─" * 65)

agent_results = []

for exc in exceptions:
    result = run_exception_agent(
        invoice_number  = exc["invoice_number"],
        reason_code     = exc["reason_code"],
        invoice_amount  = float(exc["invoice_amount"] or 0),
        po_amount       = float(exc["po_total_amount"] or 0),
        gr_number       = exc["gr_number"],
        invoice_vendor  = exc["invoice_vendor"],
        po_vendor       = exc["po_vendor_name"]
    )

    agent_results.append({
        "invoice_number":    exc["invoice_number"],
        "reason_code":       exc["reason_code"],
        "agent_verdict":     result["verdict"],
        "agent_resolution":  result["resolution"],
        "agent_confidence":  result["confidence"],
        "recommended_action":result["recommended_action"],
        "processed_at":      datetime.now().isoformat(),
    })

    icon = "✅" if result["verdict"] == "RESOLVED" else "⚠️ "
    print(f"  {icon} {exc['invoice_number']:25s} | "
          f"{exc['reason_code']:20s} | "
          f"conf: {result['confidence']:.2f} | "
          f"{result['verdict']}")
    time.sleep(0.5)

print("\n" + "─" * 65)
resolved  = sum(1 for r in agent_results if r["agent_verdict"] == "RESOLVED")
escalated = len(agent_results) - resolved
print(f"✅ Auto-resolved : {resolved}/{len(exceptions)}")
print(f"⚠️  Escalated    : {escalated}/{len(exceptions)}")
print(f"Auto-resolve rate: {round(resolved/len(exceptions)*100,1)}%")

In [0]:
# Write agent verdicts to Gold
schema = T.StructType([
    T.StructField("invoice_number",    T.StringType()),
    T.StructField("reason_code",       T.StringType()),
    T.StructField("agent_verdict",     T.StringType()),
    T.StructField("agent_resolution",  T.StringType()),
    T.StructField("agent_confidence",  T.DoubleType()),
    T.StructField("recommended_action",T.StringType()),
    T.StructField("processed_at",      T.StringType()),
])

(spark.createDataFrame(agent_results, schema=schema)
      .write.format("delta")
      .mode("overwrite")
      .saveAsTable(f"{CATALOG}.gold.agent_exception_verdicts"))

print(f"✅ Written to {CATALOG}.gold.agent_exception_verdicts\n")

# Breakdown by reason code
print("Agent verdicts by reason code:")
display(
    spark.table(f"{CATALOG}.gold.agent_exception_verdicts")
         .groupBy("reason_code", "agent_verdict")
         .agg(F.count("*").alias("count"),
              F.round(F.avg("agent_confidence"), 2).alias("avg_conf"))
         .orderBy("reason_code", "agent_verdict")
)

In [0]:
gt = spark.read.csv(GT_PATH, header=True, inferSchema=True)

agent_eval = (
    gt.join(
        spark.table(f"{CATALOG}.gold.agent_exception_verdicts")
             .select("invoice_number", "agent_verdict", "agent_confidence"),
        on="invoice_number", how="inner"
    )
    .withColumn("expected_status",
        F.when(F.col("expected_overall_match") == True,
               F.lit("APPROVED")).otherwise(F.lit("EXCEPTION"))
    )
    .withColumn("agent_status",
        F.when(F.col("agent_verdict") == "RESOLVED",
               F.lit("APPROVED")).otherwise(F.lit("EXCEPTION"))
    )
    .withColumn("correct",
        F.col("agent_status") == F.col("expected_status")
    )
)

total   = agent_eval.count()
correct = agent_eval.filter("correct = true").count()
print(f"Agent accuracy on exceptions: {correct}/{total} = "
      f"{round(correct/total*100,1)}%")

print("\nPer-scenario breakdown:")
display(
    agent_eval
    .groupBy("scenario", "agent_verdict")
    .agg(F.count("*").alias("count"),
         F.sum(F.col("correct").cast("int")).alias("correct"))
    .orderBy("scenario")
)

print("\nWrong decisions (if any):")
display(
    agent_eval.filter("correct = false")
              .select("invoice_number", "scenario",
                      "expected_status", "agent_status",
                      "agent_verdict", "agent_confidence")
)

In [0]:
mlflow.set_experiment(
    "/Users/shubhammudliar27@outlook.com/p2p_exception_agent"
)

with mlflow.start_run(
    run_name=f"exception_agent_{datetime.now().strftime('%Y%m%d_%H%M')}"
) as run:

    mlflow.log_param("llm_endpoint",       LLM)
    mlflow.log_param("agent_framework",    "databricks_native_ai_query")
    mlflow.log_param("tools_count",        5)
    mlflow.log_param("exceptions_processed", len(agent_results))

    mlflow.log_metric("total_exceptions",  len(agent_results))
    mlflow.log_metric("auto_resolved",     resolved)
    mlflow.log_metric("escalated",         escalated)
    mlflow.log_metric("auto_resolve_rate", round(resolved/len(agent_results), 4))
    mlflow.log_metric("agent_accuracy",    round(correct/total, 4))

    avg_conf = sum(r["agent_confidence"] for r in agent_results) / len(agent_results)
    mlflow.log_metric("avg_confidence",    round(avg_conf, 4))

    # Per reason code metrics
    for reason in ["EXC_DUPLICATE", "EXC_PRICE", "EXC_QTY_MISMATCH",
                   "EXC_VENDOR", "EXC_DATE"]:
        subset   = [r for r in agent_results if r["reason_code"] == reason]
        if subset:
            res_rate = sum(1 for r in subset
                          if r["agent_verdict"] == "RESOLVED") / len(subset)
            safe     = reason.lower()
            mlflow.log_metric(f"resolve_rate_{safe}", round(res_rate, 4))

    run_id = run.info.run_id

print(f"""
✅ Exception Agent logged to MLflow
{'─'*50}
  Run ID           : {run_id}
  Total exceptions : {len(agent_results)}
  Auto-resolved    : {resolved}  ({round(resolved/len(agent_results)*100,1)}%)
  Escalated        : {escalated}
  Agent accuracy   : {round(correct/total*100,1)}%
  Avg confidence   : {round(avg_conf*100,1)}%
{'─'*50}
""")